## 1: setup

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

%load_ext autoreload
%autoreload 2
from src.evaluation import evaluate

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
movies = pd.read_csv("../data/ml-latest-small/movies.csv")

## 2: the matrix

In [2]:
user_ids = np.sort(train["userId"].unique())
item_ids = np.sort(train["movieId"].unique())
user_index = {u: i for i, u in enumerate(user_ids)}
item_index = {m: i for i, m in enumerate(item_ids)}

rows = train["userId"].map(user_index)
cols = train["movieId"].map(item_index)
R = csr_matrix((train["rating"], (rows, cols)), shape=(len(user_ids), len(item_ids)))

print("Matrix shape (users x movies):", R.shape)
print("Ratings stored:", R.nnz)

Matrix shape (users x movies): (602, 3637)
Ratings stored: 63357


## 3: similarity

In [3]:
S = cosine_similarity(R.T)
print("Similarity matrix shape:", S.shape)

Similarity matrix shape: (3637, 3637)


## 4: sanity check, which movies look similar to Toy Story?

In [4]:
def similar_movies(title, n=5):
    movie_id = movies.loc[movies["title"] == title, "movieId"].iloc[0]
    i = item_index[movie_id]
    order = np.argsort(-S[i])[1:n + 1]      # [0] is the movie itself
    return movies.set_index("movieId").loc[item_ids[order], "title"]

similar_movies("Toy Story (1995)")

movieId
780          Independence Day (a.k.a. ID4) (1996)
3114                           Toy Story 2 (1999)
480                          Jurassic Park (1993)
260     Star Wars: Episode IV - A New Hope (1977)
588                                Aladdin (1992)
Name: title, dtype: str

## 5: score every movie for every user

In [5]:
scores = R @ S
print(scores.shape)

(602, 3637)


## 6: the recommender function

In [6]:
def recommend_itemknn(user_id, seen, k):
    u = user_index[user_id]
    user_scores = scores[u].copy()
    seen_idx = [item_index[m] for m in seen if m in item_index]
    user_scores[seen_idx] = -np.inf          # never recommend seen movies
    top = np.argpartition(-user_scores, k)[:k]
    top = top[np.argsort(-user_scores[top])]
    return item_ids[top].tolist()

## 7: compare with the baselines

In [7]:
r = evaluate(recommend_itemknn, history=train, target=val, k=10)
r["coverage"] = r["n_unique_items_recommended"] / len(item_ids)
pd.Series(r).round(4)

precision@k                     0.0712
recall@k                        0.1013
ndcg@k                          0.0995
users_evaluated               572.0000
n_unique_items_recommended    430.0000
coverage                        0.1182
dtype: float64

### Findings: item-based collaborative filtering (validation set, k=10)

Item-kNN with cosine similarity on raw ratings scores precision@10 0.0712, recall@10 0.1013 and NDCG@10 0.0995, beating the most-popular baseline by about 31%, 39% and 25% respectively. 30 of 602 users had no liked movie in the validation data and were excluded from scoring.

Catalogue coverage is only 11.8% (430 of 3,637 movies). Raw cosine similarity favours popular movies because they share raters with many others, so the model leans toward blockbusters (see the Toy Story neighbours above).